### Fast API demo / POC
#### Define the app

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Demo API")

class EchoRequest(BaseModel):
    text: str

class EchoResponse(BaseModel):
    length: int
    upper: str

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/echo", response_model=EchoResponse)
async def echo(req: EchoRequest):
    return EchoResponse(
        length=len(req.text),
        upper=req.text.upper(),
    )


"Call" the API using Test Client

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 1) GET /health
resp = client.get("/health")
print(resp.status_code, resp.json())

# 2) POST /echo
payload = {"text": "Hello FastAPI"}
resp = client.post("/echo", json=payload)
print(resp.status_code, resp.json())


### Simple demo of multiprocessing library
#### When to use
<ul>
<li>Work is CPU-bound (heavy math, simulation, model scoring, etc.)</li>
<li>You want to bypass the GIL and use multiple cores</li>
<li>Threads won’t help because Python bytecode still runs under one GIL</li></ul>
Note: this doesn't work in Jupyter
</br></br>
For CPU-bound workloads I use multiprocessing rather than threads, because each process gets its own interpreter and GIL.
A typical pattern is multiprocessing.Pool.map: I define a pure function, create a pool with cpu_count() workers, and map a list of inputs across it.
That gives me real parallelism on multi-core machines for heavy computations like simulations or batch scoring, instead of just overlapping I/O like with async or threads.”

In [ ]:
''' import math
from time import time

def heavy_compute(n: int) -> float:
    # Fake CPU work: sum of square roots up to n
    return sum(math.sqrt(i) for i in range(n))

nums = [5_000_00, 6_000_00, 7_000_00, 8_000_00]  # four jobs

t0 = time()
seq_results = [heavy_compute(n) for n in nums]
t1 = time()

print(f"Sequential time: {t1 - t0:.2f}s")


from multiprocessing import Pool, cpu_count

t0 = time()
with Pool(processes=cpu_count()) as pool:
    par_results = pool.map(heavy_compute, nums)
t1 = time()

print(f"Parallel time:   {t1 - t0:.2f}s")
'''

In [4]:
import math
from time import time
from concurrent.futures import ThreadPoolExecutor

def heavy_compute(n: int) -> float:
    return sum(math.sqrt(i) for i in range(n))

nums = [5_000_000, 6_000_000, 7_000_000, 8_000_000]

t0 = time()
seq_results = [heavy_compute(n) for n in nums]
t1 = time()
print(f"Sequential: {t1 - t0:.2f}s")

t0 = time()
with ThreadPoolExecutor(max_workers=4) as ex:
    par_results = list(ex.map(heavy_compute, nums))
t1 = time()
print(f"Threaded:   {t1 - t0:.2f}s")


Sequential: 0.77s
Threaded:   0.74s


### Simple demo of dealing with I/O bound tasks
Do a dummy task which waits 1 sec to complete. Aggregate results after all have executed.

In [14]:
import time

def fake_io_task(i: int) -> int:
    print(f"Start task {i}")
    time.sleep(1)  # simulate I/O wait
    print(f"End task {i}")
    return i * 10  # pretend this is some useful response

def aggregate_results(results):
    print("All tasks done. Aggregating...")
    total = sum(results)
    print(f"Aggregated total: {total}")
    return total

# Parameters
nums = [1, 2, 3, 4, 5, 6, 7, 8] # simulate 8 I/O tasks

In [15]:
# Run in sequence
t0 = time.time()
seq_results = [fake_io_task(n) for n in nums]
agg_seq_results = aggregate_results(seq_results)
t1 = time.time()

print("Sequential results:", agg_seq_results)
print(f"Sequential time: {t1 - t0:.2f}s")


Start task 1
End task 1
Start task 2
End task 2
Start task 3
End task 3
Start task 4
End task 4
Start task 5
End task 5
Start task 6
End task 6
Start task 7
End task 7
Start task 8
End task 8
All tasks done. Aggregating...
Aggregated total: 360
Sequential results: 360
Sequential time: 8.03s


In [18]:
# Run in parallel

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

t0 = time.time()
parallel_results = []

# --- Stage 1: run I/O-bound tasks in parallel ---
# Note: set max_workers based on various backend limits (e.g., DB connections, API rate limits, etc.)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = [ex.submit(fake_io_task, n) for n in nums]
    for fut in as_completed(futures):
        parallel_results.append(fut.result())

# --- Stage 2: only run AFTER all futures completed ---
agg_parallel_results = aggregate_results(parallel_results)

t1 = time.time()
print(f"Total wall time: {t1 - t0:.2f}s")


Start task 1Start task 2

Start task 3
Start task 4
Start task 5
Start task 6
Start task 7
Start task 8
End task 5End task 8
End task 4

End task 2
End task 1
End task 3
End task 6
End task 7
All tasks done. Aggregating...
Aggregated total: 360
Total wall time: 1.01s
